# Phase 3: Pretrained Genomic Embedding Experiments

In this phase, we use a pretrained foundation model (`InstaDeepAI/nucleotide-transformer-50m-1000g`) to extract dense vector representations of our DNA sequences. We will then train classical interpretable models (Logistic Regression, Random Forest, XGBoost) on these embeddings and compare their performance against the Phase 1 $k$-mer baseline and Phase 2 CNNs.

## 1. Environment Setup & Repository Integration
This cell pulls the latest codebase from our GitHub repository, adds the source code to the Python path, and dynamically installs missing requirements like `xgboost`, `transformers`, and `torch`.

In [ ]:
import os
import sys
import subprocess
import importlib
import time
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, log_loss

# 1. Git pull or clone codebase
repo_dir = "/kaggle/working/interpretable-regulatory-genomics"
if not os.path.exists(repo_dir):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/PxA-Labs/interpretable-regulatory-genomics.git", repo_dir], check=True)
else:
    print("Repository exists. Pulling latest updates...")
    subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

# 2. Add repository directory to python search path
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
if os.path.join(repo_dir, "src") not in sys.path:
    sys.path.insert(0, os.path.join(repo_dir, "src"))

# 3. Auto-install missing libraries
def install_and_import(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Library '{package_name}' not found. Installing...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package_name], check=True)

install_and_import("xgboost")
install_and_import("transformers")
install_and_import("torch")

## 2. Dynamic Package Module Loading
This cell dynamically imports and reloads all of our custom `src` modules to ensure we are running the freshest code from the repository.

In [ ]:
import src
import src.data
import src.data.download
import src.data.parse_encode
import src.data.sequence_extractor
import src.data.negative_sampling
import src.features
import src.features.embeddings
import src.models
import src.models.base_model
import src.models.logistic
import src.models.tree_ensemble
import src.models.registry

# Reload modules
for module in [
    src, src.data, src.data.download, src.data.parse_encode,
    src.data.sequence_extractor, src.data.negative_sampling,
    src.features, src.features.embeddings, src.models, src.models.base_model,
    src.models.logistic, src.models.tree_ensemble, src.models.registry
]:
    importlib.reload(module)

print("Modules loaded.")

## 3A. Directory Initialization & Data Setup
This cell sets up our working directories (`data/raw`, `data/processed`, etc.) and automatically detects if the full HG38 reference genome is mounted on Kaggle. If not, it falls back to a test mode using just Chromosome 19.

In [ ]:
raw_dir = "/kaggle/working/data/raw"
processed_dir = "/kaggle/working/data/processed"
reference_dir = "/kaggle/working/data/reference"

for path in [raw_dir, processed_dir, reference_dir]:
    os.makedirs(path, exist_ok=True)

def detect_fasta_source():
    input_base = "/kaggle/input"
    if os.path.exists(input_base):
        for root, dirs, files in os.walk(input_base):
            has_chrom_fasta = any(
                (f.startswith("chr") and (f.endswith(".fa") or f.endswith(".fa.gz") or f.endswith(".fasta") or f.endswith(".fasta.gz")))
                for f in files
            )
            if has_chrom_fasta:
                print(f"Detected mounted reference genome at: {root}")
                return root, False
    
    print("Mounted hg38 dataset not found. Running in Test mode (using chr19)...")
    return reference_dir, True

fasta_source, is_test_mode = detect_fasta_source()

## 3B. Data Extraction Pipeline
This cell executes our core data pipeline: it downloads the ENCODE annotation files, resizes the regulatory regions to 1000bp, extracts the raw DNA sequences from the FASTA file, and samples GC-matched non-coding sequences for the negative class.

In [ ]:
ccres_raw_path = src.data.download.download_encode_ccres(raw_dir)

if is_test_mode:
    print("Downloading chr19 reference sequence for test execution...")
    src.data.download.download_hg38_chromosome("chr19", reference_dir)
    chromosomes_to_use = ["chr19"]
else:
    chromosomes_to_use = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

parsed_bed_path = os.path.join(processed_dir, "k562_active_ccres.bed")
src.data.parse_encode.parse_and_resize_ccres(
    input_path=ccres_raw_path,
    output_path=parsed_bed_path,
    element_types=['PLS', 'dELS'],
    target_length=1000,
    chromosomes=chromosomes_to_use
)

extracted_tsv_path = os.path.join(processed_dir, "k562_active_ccres_sequences.tsv")
src.data.sequence_extractor.extract_sequences(
    bed_path=parsed_bed_path,
    fasta_source=fasta_source,
    output_path=extracted_tsv_path,
    max_n_fraction=0.10
)

combined_dataset_path = os.path.join(processed_dir, "k562_combined_dataset.tsv")
src.data.negative_sampling.build_negative_dataset(
    pos_tsv_path=extracted_tsv_path,
    fasta_source=fasta_source,
    output_path=combined_dataset_path,
    target_length=1000
)

## 4. Train/Test Split (Raw Sequences)
This cell loads the generated positive and negative sequences into memory and partitions them. It uses a chromosome-holdout strategy (testing on chr19) if the full genome is loaded, otherwise it uses an 80/20 random stratified split.

In [ ]:
df_dataset = pd.read_csv(combined_dataset_path, sep='\t')

if is_test_mode:
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(
        df_dataset, test_size=0.2, random_state=42, stratify=df_dataset['label']
    )
else:
    chroms = df_dataset['chrom'].values
    train_mask = ~np.isin(chroms, ['chr19', 'chr20', 'chr21', 'chr22'])
    test_mask = (chroms == 'chr19')
    
    if train_mask.sum() == 0 or test_mask.sum() == 0:
        from sklearn.model_selection import train_test_split
        train_df, test_df = train_test_split(
            df_dataset, test_size=0.2, random_state=42, stratify=df_dataset['label']
        )
    else:
        train_df = df_dataset[train_mask]
        test_df = df_dataset[test_mask]

train_seqs = train_df['sequence'].tolist()
test_seqs = test_df['sequence'].tolist()
y_train = train_df['label'].values
y_test = test_df['label'].values

print(f"Train size: {len(train_seqs)}")
print(f"Test size: {len(test_seqs)}")

## 5. Extract Pretrained Embeddings
This cell instantiates our `PretrainedEmbedder` to download the `Nucleotide Transformer 50M` weights from HuggingFace and processes the raw DNA strings in batches to generate deep multidimensional feature vectors. *(Ensure GPU T4 and Internet are enabled in Kaggle Settings).* 

In [ ]:
# Initialize Embedder
embedder = src.features.embeddings.PretrainedEmbedder(model_name="InstaDeepAI/nucleotide-transformer-50m-1000g")

# Extract features
print("Extracting training embeddings...")
X_train_emb = embedder.get_embeddings(train_seqs, batch_size=32)

print("Extracting test embeddings...")
X_test_emb = embedder.get_embeddings(test_seqs, batch_size=32)

print(f"Train Embeddings Shape: {X_train_emb.shape}")
print(f"Test Embeddings Shape: {X_test_emb.shape}")

## 6. Train Classifiers on Embeddings & Compare Results
This final cell trains three models (Logistic Regression, Random Forest, and XGBoost) using the newly extracted embeddings as features. It then prints a side-by-side comparison table including hardcoded results from our Phase 1 and Phase 2 baselines to immediately determine if the pretrained model improved performance.

In [ ]:
models = {
    "Phase 3 - Logistic Regression (Embeddings)": src.models.registry.ModelRegistry.get_model("logistic_regression"),
    "Phase 3 - Random Forest (Embeddings)": src.models.registry.ModelRegistry.get_model("random_forest", params={"n_estimators": 100}),
    "Phase 3 - XGBoost (Embeddings)": src.models.registry.ModelRegistry.get_model("xgboost", params={"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1})
}

results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    start_time = time.time()
    model.train(X_train_emb, y_train)
    elapsed = time.time() - start_time
    
    # Evaluate
    preds = model.predict(X_test_emb)
    probs = model.predict_proba(X_test_emb)
    
    auroc = roc_auc_score(y_test, probs)
    auprc = average_precision_score(y_test, probs)
    f1 = f1_score(y_test, preds)
    loss = log_loss(y_test, probs)
    
    results.append({
        "Model": name,
        "AUROC": auroc,
        "AUPRC": auprc,
        "F1-Score": f1,
        "Log Loss": loss,
    })

results_df = pd.DataFrame(results)

# Hardcoded Baseline Results from Phase 1 and 2 for easy comparison
baseline_results = [
    {"Model": "Phase 1 - Logistic Regression (k=4)", "AUROC": 0.7795, "AUPRC": 0.7423, "F1-Score": 0.7292, "Log Loss": 0.6596},
    {"Model": "Phase 1 - Random Forest (k=4)", "AUROC": 0.8680, "AUPRC": 0.8855, "F1-Score": 0.7886, "Log Loss": 0.4827},
    {"Model": "Phase 1 - XGBoost (k=4)", "AUROC": 0.8830, "AUPRC": 0.8879, "F1-Score": 0.8062, "Log Loss": 0.6011},
    {"Model": "Phase 2 - Shallow CNN (One-Hot)", "AUROC": 0.8604, "AUPRC": 0.8718, "F1-Score": 0.7802, "Log Loss": 0.4604},
    {"Model": "Phase 2 - Deep CNN (One-Hot)", "AUROC": 0.8537, "AUPRC": 0.8603, "F1-Score": 0.7909, "Log Loss": 0.5345}
]
baseline_df = pd.DataFrame(baseline_results)

# Combine all results
combined_df = pd.concat([baseline_df, results_df], ignore_index=True)

print("\n=================== GRAND COMPARISON (ALL PHASES) ===================")
display(combined_df)
print("=====================================================================\n")